# 🔄 Notebook 03c — Prophet Retreinado com Monte Carlo (2023–2025)
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Retreinar o Prophet ensemble (v5+v6) usando a série estendida de 3 anos
(2023–2024 sintético + 2025 real) gerada no notebook 03b, e comparar o MAE com o
modelo original treinado apenas com 2025.

**Hipótese:** Mais dados históricos melhoram a estimativa de sazonalidade anual
do Prophet, reduzindo o MAE especialmente nos horizontes D+5 a D+7.

**Input:** `data/processed/serie_sintetica_2023_2024.csv`
**Output:** `outputs/previsoes_volume_mc.json` + comparação de MAE

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import holidays
import json, os, pickle
from datetime import date

print("✅ Imports ok")

Importing plotly failed. Interactive plots will not work.


✅ Imports ok


In [2]:
# ── Carregar as três séries Monte Carlo (total, P2, P3) ──────────────────────
serie_total_mc = pd.read_csv('../data/processed/serie_sintetica_2023_2024.csv',
                              parse_dates=['data'])
serie_p2_mc    = pd.read_csv('../data/processed/serie_p2_mc.csv',
                              parse_dates=['data'])
serie_p3_mc    = pd.read_csv('../data/processed/serie_p3_mc.csv',
                              parse_dates=['data'])

# Renomear para o formato Prophet (ds, y)
def prep(serie):
    df = serie[['data', 'y']].rename(columns={'data': 'ds'}).copy()
    calendario = pd.DataFrame({'ds': pd.date_range('2023-01-01', '2025-12-31', freq='D')})
    df = calendario.merge(df, on='ds', how='left').fillna({'y': 0})
    df['y'] = df['y'].astype(float)
    return df.sort_values('ds').reset_index(drop=True)

serie_total = prep(serie_total_mc)
serie_p2    = prep(serie_p2_mc)
serie_p3    = prep(serie_p3_mc)

print(f'Total: {len(serie_total)} dias | média: {serie_total.y.mean():.1f}')
print(f'P2:    {len(serie_p2)} dias | média: {serie_p2.y.mean():.1f}')
print(f'P3:    {len(serie_p3)} dias | média: {serie_p3.y.mean():.1f}')
print(f'Período: {serie_total.ds.min().date()} a {serie_total.ds.max().date()}')

Total: 1097 dias | média: 68.7
P2:    1097 dias | média: 14.2
P3:    1097 dias | média: 54.5
Período: 2023-01-01 a 2025-12-31


In [3]:
# ── Feriados brasileiros 2023–2025 ────────────────────────────────────────────
br_holidays_all = holidays.Brazil(years=[2023, 2024, 2025])
feriados_df = pd.DataFrame([
    {'ds': pd.Timestamp(d), 'holiday': n}
    for d, n in br_holidays_all.items()
])

# Feriados extras que se comportam como feriado na operação
feriados_extras = pd.DataFrame([
    # Carnaval
    {'ds': pd.Timestamp('2023-02-20'), 'holiday': 'Carnaval — Segunda'},
    {'ds': pd.Timestamp('2023-02-21'), 'holiday': 'Carnaval — Terça'},
    {'ds': pd.Timestamp('2023-02-22'), 'holiday': 'Carnaval — Quarta de Cinzas'},
    {'ds': pd.Timestamp('2024-02-12'), 'holiday': 'Carnaval — Segunda'},
    {'ds': pd.Timestamp('2024-02-13'), 'holiday': 'Carnaval — Terça'},
    {'ds': pd.Timestamp('2024-02-14'), 'holiday': 'Carnaval — Quarta de Cinzas'},
    {'ds': pd.Timestamp('2025-03-03'), 'holiday': 'Carnaval — Segunda'},
    {'ds': pd.Timestamp('2025-03-04'), 'holiday': 'Carnaval — Terça'},
    {'ds': pd.Timestamp('2025-03-05'), 'holiday': 'Carnaval — Quarta de Cinzas'},
    # Corpus Christi
    {'ds': pd.Timestamp('2023-06-08'), 'holiday': 'Corpus Christi'},
    {'ds': pd.Timestamp('2024-05-30'), 'holiday': 'Corpus Christi'},
    {'ds': pd.Timestamp('2025-06-19'), 'holiday': 'Corpus Christi'},
    # Vésperas
    {'ds': pd.Timestamp('2023-12-24'), 'holiday': 'Véspera de Natal'},
    {'ds': pd.Timestamp('2023-12-31'), 'holiday': 'Véspera de Ano Novo'},
    {'ds': pd.Timestamp('2024-12-24'), 'holiday': 'Véspera de Natal'},
    {'ds': pd.Timestamp('2024-12-31'), 'holiday': 'Véspera de Ano Novo'},
    {'ds': pd.Timestamp('2025-12-24'), 'holiday': 'Véspera de Natal'},
    {'ds': pd.Timestamp('2025-12-31'), 'holiday': 'Véspera de Ano Novo'},
])

feriados_completo = pd.concat([feriados_df, feriados_extras], ignore_index=True)
feriados_completo = feriados_completo.sort_values('ds').reset_index(drop=True)

print(f'Total de feriados 2023–2025: {len(feriados_completo)}')
print(feriados_completo.head(10).to_string(index=False))

Total de feriados 2023–2025: 47
        ds                      holiday
2023-01-01 Universal Fraternization Day
2023-02-20           Carnaval — Segunda
2023-02-21             Carnaval — Terça
2023-02-22  Carnaval — Quarta de Cinzas
2023-04-07                  Good Friday
2023-04-21              Tiradentes' Day
2023-05-01                 Worker's Day
2023-06-08               Corpus Christi
2023-09-07             Independence Day
2023-10-12        Our Lady of Aparecida


In [8]:
# ── Split temporal: treino e holdout ─────────────────────────────────────────
HOLDOUT_INICIO = '2025-10-01'
HOLDOUT_FIM    = '2025-12-31'
TREINO_INICIO  = '2023-01-01'
TREINO_FIM     = '2025-09-30'

# Separar treino e holdout na série total
serie_treino_total = serie_total[serie_total['ds'] <= TREINO_FIM].copy()
serie_holdout_total = serie_total[serie_total['ds'] >= HOLDOUT_INICIO].copy()

# P2 e P3
serie_treino_p2 = serie_p2[serie_p2['ds'] <= TREINO_FIM].copy()
serie_holdout_p2 = serie_p2[serie_p2['ds'] >= HOLDOUT_INICIO].copy()

serie_treino_p3 = serie_p3[serie_p3['ds'] <= TREINO_FIM].copy()
serie_holdout_p3 = serie_p3[serie_p3['ds'] >= HOLDOUT_INICIO].copy()

print('=== Split treino / holdout ===')
print(f'Treino total:   {len(serie_treino_total)} dias | {serie_treino_total.ds.min().date()} a {serie_treino_total.ds.max().date()}')
print(f'Holdout total:  {len(serie_holdout_total)} dias | {serie_holdout_total.ds.min().date()} a {serie_holdout_total.ds.max().date()}')
print()
print(f'Holdout é 100% real (2025): {serie_holdout_total.ds.min().date() >= pd.Timestamp("2025-01-01").date()}')
print(f'Treino tem sintético:       {(serie_treino_total.ds.min().date() < pd.Timestamp("2025-01-01").date())}')

=== Split treino / holdout ===
Treino total:   1005 dias | 2023-01-01 a 2025-09-30
Holdout total:  92 dias | 2025-10-01 a 2025-12-31

Holdout é 100% real (2025): True
Treino tem sintético:       True


In [9]:
# ── Adicionar lags nas séries de treino ───────────────────────────────────────
def adicionar_lags(df):
    df = df.copy().sort_values('ds').reset_index(drop=True)
    df['lag_1d']      = df['y'].shift(1)
    df['lag_7d']      = df['y'].shift(7)
    df['rolling_7d']  = df['y'].rolling(7,  min_periods=1).mean()
    df['rolling_30d'] = df['y'].rolling(30, min_periods=1).mean()
    df['is_dia_util'] = df['ds'].dt.dayofweek.apply(
        lambda d: 0.0 if d >= 5 else 1.0
    ).astype(float)
    feriados_set = set(feriados_completo['ds'].dt.date)
    df.loc[df['ds'].dt.date.isin(feriados_set), 'is_dia_util'] = 0.0
    # Remover linhas com NaN em lag_1d e lag_7d
    df = df.dropna(subset=['lag_1d', 'lag_7d']).reset_index(drop=True)
    return df

serie_treino_total_lag = adicionar_lags(serie_treino_total)
serie_treino_p2_lag    = adicionar_lags(serie_treino_p2)
serie_treino_p3_lag    = adicionar_lags(serie_treino_p3)

print(f'Treino total: {len(serie_treino_total_lag)} dias')
print(f'Treino P2:    {len(serie_treino_p2_lag)} dias')
print(f'Treino P3:    {len(serie_treino_p3_lag)} dias')
print()
print('Amostra treino total:')
print(serie_treino_total_lag.sample(3).to_string(index=False))

Treino total: 998 dias
Treino P2:    998 dias
Treino P3:    998 dias

Amostra treino total:
        ds    y  lag_1d  lag_7d  rolling_7d  rolling_30d  is_dia_util
2025-08-28 84.0    88.0    98.0   74.571429    78.833333          1.0
2023-04-28 37.0    75.0    80.0   56.142857    69.766667          1.0
2023-12-26 69.0    75.0    65.0   54.142857    52.933333          1.0


In [10]:
# ── Função auxiliar para preencher regressores no futuro ──────────────────────
def preencher_regressores(futuro, serie_lag, regressores):
    lookup = serie_lag.drop_duplicates('ds').set_index('ds')
    media_recente = serie_lag.tail(30)
    for reg in regressores:
        futuro[reg] = futuro['ds'].map(lookup[reg]).fillna(media_recente[reg].mean())
    return futuro

REGS_V5 = ['lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d']
REGS_V6 = ['lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d', 'is_dia_util']
feriados_set = set(feriados_completo['ds'].dt.date)

PROPHET_PARAMS = dict(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    holidays=feriados_completo,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.01,
    holidays_prior_scale=8.0,
    seasonality_prior_scale=10.0,
)

# ── v5 Total ──────────────────────────────────────────────────────────────────
m_total_v5 = Prophet(**PROPHET_PARAMS)
for r in REGS_V5: m_total_v5.add_regressor(r)
m_total_v5.fit(serie_treino_total_lag)

futuro_v5 = m_total_v5.make_future_dataframe(periods=7, freq='D')
futuro_v5 = preencher_regressores(futuro_v5, serie_treino_total_lag, REGS_V5)
prev_total_v5 = m_total_v5.predict(futuro_v5)

# ── v6 Total ──────────────────────────────────────────────────────────────────
m_total_v6 = Prophet(**PROPHET_PARAMS)
for r in REGS_V6: m_total_v6.add_regressor(r)
m_total_v6.fit(serie_treino_total_lag)

futuro_v6 = m_total_v6.make_future_dataframe(periods=7, freq='D')
futuro_v6 = preencher_regressores(futuro_v6, serie_treino_total_lag, REGS_V5)
futuro_v6['is_dia_util'] = futuro_v6['ds'].dt.dayofweek.apply(lambda d: 0.0 if d >= 5 else 1.0)
futuro_v6.loc[futuro_v6['ds'].dt.date.isin(feriados_set), 'is_dia_util'] = 0.0
prev_total_v6 = m_total_v6.predict(futuro_v6)

print('✅ v5 e v6 Total treinados')
print()
print('=== Previsão D+1 a D+7 ===')
for i, (r5, r6) in enumerate(zip(prev_total_v5.tail(7).itertuples(),
                                   prev_total_v6.tail(7).itertuples())):
    print(f'D+{i+1} {r5.ds.date()}: v5={r5.yhat:.1f} | v6={r6.yhat:.1f}')

04:31:18 - cmdstanpy - INFO - Chain [1] start processing
04:31:18 - cmdstanpy - INFO - Chain [1] done processing
04:31:18 - cmdstanpy - INFO - Chain [1] start processing
04:31:18 - cmdstanpy - INFO - Chain [1] done processing


✅ v5 e v6 Total treinados

=== Previsão D+1 a D+7 ===
D+1 2025-10-01: v5=81.3 | v6=82.1
D+2 2025-10-02: v5=79.4 | v6=80.1
D+3 2025-10-03: v5=69.2 | v6=69.2
D+4 2025-10-04: v5=52.7 | v6=51.5
D+5 2025-10-05: v5=62.5 | v6=61.5
D+6 2025-10-06: v5=85.1 | v6=85.7
D+7 2025-10-07: v5=84.1 | v6=85.0


In [11]:
# ── Validar v5 e v6 no holdout de outubro/2025 ───────────────────────────────
# Pegar os reais do holdout para comparar
real_holdout = serie_holdout_total[['ds', 'y']].copy()
real_holdout = real_holdout[real_holdout['ds'] <= '2025-10-07'].copy()

# Previsão dos primeiros 7 dias do holdout
prev_holdout_v5 = prev_total_v5[
    prev_total_v5['ds'].isin(real_holdout['ds'])
][['ds', 'yhat']].copy()

prev_holdout_v6 = prev_total_v6[
    prev_total_v6['ds'].isin(real_holdout['ds'])
][['ds', 'yhat']].copy()

# Merge
comp = real_holdout.merge(
    prev_holdout_v5.rename(columns={'yhat': 'v5'}), on='ds'
).merge(
    prev_holdout_v6.rename(columns={'yhat': 'v6'}), on='ds'
)
comp['erro_v5'] = (comp['v5'] - comp['y']).abs()
comp['erro_v6'] = (comp['v6'] - comp['y']).abs()

print('=== Holdout D+1 a D+7 — Real vs Previsto ===')
print(f'{"Data":<12} {"Real":>6} {"v5":>7} {"v6":>7} {"|err v5|":>10} {"|err v6|":>10}')
print('-' * 55)
for _, r in comp.iterrows():
    print(f'{str(r.ds.date()):<12} {r.y:>6.0f} {r.v5:>7.1f} {r.v6:>7.1f} {r.erro_v5:>10.1f} {r.erro_v6:>10.1f}')
print('-' * 55)
print(f'{"MAE":<12} {"":>6} {"":>7} {"":>7} {comp.erro_v5.mean():>10.1f} {comp.erro_v6.mean():>10.1f}')

=== Holdout D+1 a D+7 — Real vs Previsto ===
Data           Real      v5      v6   |err v5|   |err v6|
-------------------------------------------------------
2025-10-01       81    81.3    82.1        0.3        1.1
2025-10-02      110    79.4    80.1       30.6       29.9
2025-10-03       98    69.2    69.2       28.8       28.8
2025-10-04       41    52.7    51.5       11.7       10.5
2025-10-05       13    62.5    61.5       49.5       48.5
2025-10-06       77    85.1    85.7        8.1        8.7
2025-10-07       96    84.1    85.0       11.9       11.0
-------------------------------------------------------
MAE                                       20.1       19.8


In [12]:
# ── Cross-validation — v5 e v6 Total ─────────────────────────────────────────
# initial=540d cobre ~18 meses de treino (suficiente com 3 anos de dados)
print('Rodando cross-validation v5...')
cv_v5 = cross_validation(
    m_total_v5,
    initial='540 days',
    period='30 days',
    horizon='7 days',
    parallel=None,
)
met_v5 = performance_metrics(cv_v5)

print('Rodando cross-validation v6...')
cv_v6 = cross_validation(
    m_total_v6,
    initial='540 days',
    period='30 days',
    horizon='7 days',
    parallel=None,
)
met_v6 = performance_metrics(cv_v6)

print()
print('=== MAE cross-validation — Monte Carlo ===')
print(f'{"Horizonte":<10} {"MAE v5":>8} {"MAE v6":>8} {"Vencedor":>10}')
print('-' * 40)
for h in range(1, 8):
    m5 = met_v5[met_v5['horizon'].dt.days == h]['mae'].values[0]
    m6 = met_v6[met_v6['horizon'].dt.days == h]['mae'].values[0]
    venc = 'v5 ✓' if m5 <= m6 else 'v6 ✓'
    print(f'D+{h:<9} {m5:>8.2f} {m6:>8.2f} {venc:>10}')

Rodando cross-validation v5...


  0%|          | 0/15 [00:00<?, ?it/s]04:33:46 - cmdstanpy - INFO - Chain [1] start processing
04:33:46 - cmdstanpy - INFO - Chain [1] done processing
04:33:47 - cmdstanpy - INFO - Chain [1] start processing
04:33:47 - cmdstanpy - INFO - Chain [1] done processing
 13%|█▎        | 2/15 [00:00<00:01, 11.52it/s]04:33:47 - cmdstanpy - INFO - Chain [1] start processing
04:33:47 - cmdstanpy - INFO - Chain [1] done processing
04:33:47 - cmdstanpy - INFO - Chain [1] start processing
04:33:47 - cmdstanpy - INFO - Chain [1] done processing
 27%|██▋       | 4/15 [00:00<00:00, 11.27it/s]04:33:47 - cmdstanpy - INFO - Chain [1] start processing
04:33:47 - cmdstanpy - INFO - Chain [1] done processing
04:33:47 - cmdstanpy - INFO - Chain [1] start processing
04:33:47 - cmdstanpy - INFO - Chain [1] done processing
 40%|████      | 6/15 [00:00<00:00,  9.75it/s]04:33:47 - cmdstanpy - INFO - Chain [1] start processing
04:33:47 - cmdstanpy - INFO - Chain [1] done processing
04:33:47 - cmdstanpy - INFO - Cha

Rodando cross-validation v6...


  0%|          | 0/15 [00:00<?, ?it/s]04:33:48 - cmdstanpy - INFO - Chain [1] start processing
04:33:48 - cmdstanpy - INFO - Chain [1] done processing
04:33:48 - cmdstanpy - INFO - Chain [1] start processing
04:33:48 - cmdstanpy - INFO - Chain [1] done processing
 13%|█▎        | 2/15 [00:00<00:01, 11.40it/s]04:33:48 - cmdstanpy - INFO - Chain [1] start processing
04:33:48 - cmdstanpy - INFO - Chain [1] done processing
04:33:48 - cmdstanpy - INFO - Chain [1] start processing
04:33:48 - cmdstanpy - INFO - Chain [1] done processing
 27%|██▋       | 4/15 [00:00<00:00, 11.89it/s]04:33:48 - cmdstanpy - INFO - Chain [1] start processing
04:33:48 - cmdstanpy - INFO - Chain [1] done processing
04:33:48 - cmdstanpy - INFO - Chain [1] start processing
04:33:49 - cmdstanpy - INFO - Chain [1] done processing
 40%|████      | 6/15 [00:00<00:00, 11.96it/s]04:33:49 - cmdstanpy - INFO - Chain [1] start processing
04:33:49 - cmdstanpy - INFO - Chain [1] done processing
04:33:49 - cmdstanpy - INFO - Cha


=== MAE cross-validation — Monte Carlo ===
Horizonte    MAE v5   MAE v6   Vencedor
----------------------------------------
D+1            14.61    14.62       v5 ✓
D+2            13.01    12.80       v6 ✓
D+3            12.54    12.48       v6 ✓
D+4            12.45    12.01       v6 ✓
D+5            19.29    18.88       v6 ✓
D+6            18.20    18.08       v6 ✓
D+7            15.27    15.27       v6 ✓


In [13]:
# ── Adicionar lags P2 e P3 ───────────────────────────────────────────────────
serie_treino_p2_lag = adicionar_lags(serie_treino_p2)
serie_treino_p3_lag = adicionar_lags(serie_treino_p3)

# ── v5 P2 ────────────────────────────────────────────────────────────────────
m_p2_v5 = Prophet(**PROPHET_PARAMS)
for r in REGS_V5: m_p2_v5.add_regressor(r)
m_p2_v5.fit(serie_treino_p2_lag)
futuro_p2_v5 = m_p2_v5.make_future_dataframe(periods=7, freq='D')
futuro_p2_v5 = preencher_regressores(futuro_p2_v5, serie_treino_p2_lag, REGS_V5)
prev_p2_v5 = m_p2_v5.predict(futuro_p2_v5)

# ── v6 P2 ────────────────────────────────────────────────────────────────────
m_p2_v6 = Prophet(**PROPHET_PARAMS)
for r in REGS_V6: m_p2_v6.add_regressor(r)
m_p2_v6.fit(serie_treino_p2_lag)
futuro_p2_v6 = m_p2_v6.make_future_dataframe(periods=7, freq='D')
futuro_p2_v6 = preencher_regressores(futuro_p2_v6, serie_treino_p2_lag, REGS_V5)
futuro_p2_v6['is_dia_util'] = futuro_p2_v6['ds'].dt.dayofweek.apply(lambda d: 0.0 if d >= 5 else 1.0)
futuro_p2_v6.loc[futuro_p2_v6['ds'].dt.date.isin(feriados_set), 'is_dia_util'] = 0.0
prev_p2_v6 = m_p2_v6.predict(futuro_p2_v6)

# ── v5 P3 ────────────────────────────────────────────────────────────────────
m_p3_v5 = Prophet(**PROPHET_PARAMS)
for r in REGS_V5: m_p3_v5.add_regressor(r)
m_p3_v5.fit(serie_treino_p3_lag)
futuro_p3_v5 = m_p3_v5.make_future_dataframe(periods=7, freq='D')
futuro_p3_v5 = preencher_regressores(futuro_p3_v5, serie_treino_p3_lag, REGS_V5)
prev_p3_v5 = m_p3_v5.predict(futuro_p3_v5)

# ── v6 P3 ────────────────────────────────────────────────────────────────────
m_p3_v6 = Prophet(**PROPHET_PARAMS)
for r in REGS_V6: m_p3_v6.add_regressor(r)
m_p3_v6.fit(serie_treino_p3_lag)
futuro_p3_v6 = m_p3_v6.make_future_dataframe(periods=7, freq='D')
futuro_p3_v6 = preencher_regressores(futuro_p3_v6, serie_treino_p3_lag, REGS_V5)
futuro_p3_v6['is_dia_util'] = futuro_p3_v6['ds'].dt.dayofweek.apply(lambda d: 0.0 if d >= 5 else 1.0)
futuro_p3_v6.loc[futuro_p3_v6['ds'].dt.date.isin(feriados_set), 'is_dia_util'] = 0.0
prev_p3_v6 = m_p3_v6.predict(futuro_p3_v6)

print('✅ v5 e v6 treinados para P2 e P3')
print()
print('=== Previsão D+1 a D+7 ===')
print(f'{"Dia":<6} {"P2 v5":>7} {"P2 v6":>7} {"P3 v5":>7} {"P3 v6":>7}')
print('-'*35)
for i, (r2_5, r2_6, r3_5, r3_6) in enumerate(zip(
    prev_p2_v5.tail(7).itertuples(),
    prev_p2_v6.tail(7).itertuples(),
    prev_p3_v5.tail(7).itertuples(),
    prev_p3_v6.tail(7).itertuples(),
)):
    print(f'D+{i+1:<4} {r2_5.yhat:>7.1f} {r2_6.yhat:>7.1f} {r3_5.yhat:>7.1f} {r3_6.yhat:>7.1f}')

04:34:41 - cmdstanpy - INFO - Chain [1] start processing
04:34:41 - cmdstanpy - INFO - Chain [1] done processing
04:34:41 - cmdstanpy - INFO - Chain [1] start processing
04:34:41 - cmdstanpy - INFO - Chain [1] done processing
04:34:41 - cmdstanpy - INFO - Chain [1] start processing
04:34:41 - cmdstanpy - INFO - Chain [1] done processing
04:34:41 - cmdstanpy - INFO - Chain [1] start processing
04:34:41 - cmdstanpy - INFO - Chain [1] done processing


✅ v5 e v6 treinados para P2 e P3

=== Previsão D+1 a D+7 ===
Dia      P2 v5   P2 v6   P3 v5   P3 v6
-----------------------------------
D+1       15.2    15.3    66.9    66.2
D+2       14.8    14.8    65.3    64.6
D+3       12.4    12.5    56.8    55.9
D+4       10.6    10.6    41.3    40.0
D+5       12.5    12.4    49.4    48.1
D+6       15.7    15.8    69.8    69.1
D+7       15.2    15.3    69.5    68.6


In [14]:
# ── Ensemble v5+v6 para total, P2 e P3 ───────────────────────────────────────
def ensemble(prev_v5, prev_v6, n=7):
    rows = []
    tail_v5 = prev_v5.tail(n).reset_index(drop=True)
    tail_v6 = prev_v6.tail(n).reset_index(drop=True)
    for i in range(n):
        rows.append({
            'ds':         tail_v5.loc[i, 'ds'],
            'yhat':       round((tail_v5.loc[i, 'yhat'] + tail_v6.loc[i, 'yhat']) / 2, 1),
            'yhat_lower': round((tail_v5.loc[i, 'yhat_lower'] + tail_v6.loc[i, 'yhat_lower']) / 2, 1),
            'yhat_upper': round((tail_v5.loc[i, 'yhat_upper'] + tail_v6.loc[i, 'yhat_upper']) / 2, 1),
        })
    return pd.DataFrame(rows)

ens_total = ensemble(prev_total_v5, prev_total_v6)
ens_p2    = ensemble(prev_p2_v5,    prev_p2_v6)
ens_p3    = ensemble(prev_p3_v5,    prev_p3_v6)

print('=== Ensemble D+1 a D+7 ===')
print(f'{"Dia":<6} {"Total":>7} {"P2":>6} {"P3":>6}')
print('-' * 26)
for i, (rt, r2, r3) in enumerate(zip(
    ens_total.itertuples(),
    ens_p2.itertuples(),
    ens_p3.itertuples(),
)):
    print(f'D+{i+1:<4} {rt.yhat:>7.1f} {r2.yhat:>6.1f} {r3.yhat:>6.1f}')

# ── Exportar JSON ─────────────────────────────────────────────────────────────
output = {
    "modelo": "prophet_ensemble_monte_carlo",
    "gerado_em": date.today().strftime('%Y-%m-%d'),
    "periodo_treino": f"2023-01-08 a {TREINO_FIM}",
    "holdout": f"{HOLDOUT_INICIO} a {HOLDOUT_FIM}",
    "mae_holdout": {"v5": 20.1, "v6": 19.8},
    "d1": {
        "total": int(max(0, round(ens_total.iloc[0]['yhat']))),
        "p2":    int(max(0, round(ens_p2.iloc[0]['yhat']))),
        "p3":    int(max(0, round(ens_p3.iloc[0]['yhat']))),
    },
    "d7": {
        "total": int(max(0, round(ens_total.iloc[6]['yhat']))),
        "p2":    int(max(0, round(ens_p2.iloc[6]['yhat']))),
        "p3":    int(max(0, round(ens_p3.iloc[6]['yhat']))),
    },
    "serie": [
        {
            "ds":         rt.ds.strftime('%Y-%m-%d'),
            "total":      max(0, rt.yhat),
            "total_low":  max(0, rt.yhat_lower),
            "total_high": max(0, rt.yhat_upper),
            "P2":         max(0, r2.yhat),
            "P3":         max(0, r3.yhat),
        }
        for rt, r2, r3 in zip(
            ens_total.itertuples(),
            ens_p2.itertuples(),
            ens_p3.itertuples(),
        )
    ],
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/previsoes_volume_mc.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print()
print('✅ previsoes_volume_mc.json exportado')

=== Ensemble D+1 a D+7 ===
Dia      Total     P2     P3
--------------------------
D+1       81.7   15.2   66.5
D+2       79.8   14.8   64.9
D+3       69.2   12.5   56.3
D+4       52.1   10.6   40.7
D+5       62.0   12.5   48.7
D+6       85.4   15.8   69.4
D+7       84.6   15.3   69.0

✅ previsoes_volume_mc.json exportado


In [15]:
# ── Ensemble adaptativo MC + Original ────────────────────────────────────────
# MAE por horizonte de cada modelo (do nb03 e do 03c)
# Quanto menor o MAE, maior o peso desse modelo naquele horizonte

mae_original = {1: 17.06, 2: 10.08, 3: 8.25, 4: 12.31, 5: 9.66, 6: 20.84, 7: 14.14}
mae_mc       = {1: 14.61, 2: 12.80, 3: 12.48, 4: 12.01, 5: 18.88, 6: 18.08, 7: 15.27}

# Peso inversamente proporcional ao MAE
# Quanto menor o MAE, maior o peso
def calcular_pesos(mae_a, mae_b):
    pesos = {}
    for h in range(1, 8):
        inv_a = 1 / mae_a[h]
        inv_b = 1 / mae_b[h]
        total = inv_a + inv_b
        pesos[h] = {'mc': inv_a / total, 'orig': inv_b / total}
    return pesos

pesos = calcular_pesos(mae_mc, mae_original)

print('=== Pesos por horizonte ===')
print(f'{"Horizonte":<10} {"Peso MC":>9} {"Peso Orig":>10} {"Favorito":>10}')
print('-' * 42)
for h in range(1, 8):
    p_mc   = pesos[h]['mc']
    p_orig = pesos[h]['orig']
    fav    = 'MC' if p_mc > p_orig else 'Original'
    print(f'D+{h:<8} {p_mc:>9.3f} {p_orig:>10.3f} {fav:>10}')

=== Pesos por horizonte ===
Horizonte    Peso MC  Peso Orig   Favorito
------------------------------------------
D+1            0.539      0.461         MC
D+2            0.441      0.559   Original
D+3            0.398      0.602   Original
D+4            0.506      0.494         MC
D+5            0.338      0.662   Original
D+6            0.535      0.465         MC
D+7            0.481      0.519   Original


In [17]:
# ── Carregar previsões do modelo original (nb03) ──────────────────────────────
with open('../outputs/previsoes_volume.json', 'r') as f:
    prev_orig = json.load(f)

# Estrutura: {'total': {'serie_7d': [...]}, 'p2': {...}, 'p3': {...}}
orig_total = {d['ds']: d['yhat'] for d in prev_orig['total']['serie_7d']}
orig_p2    = {d['ds']: d['yhat'] for d in prev_orig['p2']['serie_7d']}
orig_p3    = {d['ds']: d['yhat'] for d in prev_orig['p3']['serie_7d']}

# ── Ensemble adaptativo por horizonte ────────────────────────────────────────
serie_final = []
for i, (rt, r2, r3) in enumerate(zip(
    ens_total.itertuples(),
    ens_p2.itertuples(),
    ens_p3.itertuples(),
)):
    h      = i + 1
    ds_str = rt.ds.strftime('%Y-%m-%d')
    p_mc   = pesos[h]['mc']
    p_orig = pesos[h]['orig']

    mc_total = rt.yhat
    mc_p2    = r2.yhat
    mc_p3    = r3.yhat

    ot = orig_total.get(ds_str, mc_total)
    op2 = orig_p2.get(ds_str, mc_p2)
    op3 = orig_p3.get(ds_str, mc_p3)

    serie_final.append({
        'ds':        ds_str,
        'horizonte': f'D+{h}',
        'total':     round(max(0, p_mc * mc_total + p_orig * ot), 1),
        'P2':        round(max(0, p_mc * mc_p2    + p_orig * op2), 1),
        'P3':        round(max(0, p_mc * mc_p3    + p_orig * op3), 1),
        'peso_mc':   round(p_mc, 3),
        'peso_orig': round(p_orig, 3),
    })

# ── Exportar JSON final ───────────────────────────────────────────────────────
output_final = {
    "modelo": "prophet_ensemble_adaptativo_mc_original",
    "gerado_em": date.today().strftime('%Y-%m-%d'),
    "descricao": "Ensemble adaptativo por horizonte — pesos inversamente proporcionais ao MAE",
    "mae_cv": {
        "mc_v6":    {f'D+{h}': round(met_v6[met_v6['horizon'].dt.days==h]['mae'].values[0], 2) for h in range(1,8)},
        "original": mae_original,
    },
    "pesos": {f'D+{h}': {'mc': round(pesos[h]['mc'],3), 'orig': round(pesos[h]['orig'],3)} for h in range(1,8)},
    "d1": {
        "total": int(round(serie_final[0]['total'])),
        "p2":    int(round(serie_final[0]['P2'])),
        "p3":    int(round(serie_final[0]['P3'])),
    },
    "d7": {
        "total": int(round(serie_final[6]['total'])),
        "p2":    int(round(serie_final[6]['P2'])),
        "p3":    int(round(serie_final[6]['P3'])),
    },
    "serie": serie_final,
}

with open('../outputs/previsoes_volume_mc.json', 'w', encoding='utf-8') as f:
    json.dump(output_final, f, ensure_ascii=False, indent=2)

print('=== Ensemble Adaptativo Final ===')
print(f'{"Dia":<6} {"Data":<12} {"Total":>7} {"P2":>5} {"P3":>6} {"Peso MC":>8}')
print('-' * 46)
for r in serie_final:
    print(f'{r["horizonte"]:<6} {r["ds"]:<12} {r["total"]:>7} {r["P2"]:>5} {r["P3"]:>6} {r["peso_mc"]:>8.3f}')
print()
print('✅ previsoes_volume_mc.json exportado')

=== Ensemble Adaptativo Final ===
Dia    Data           Total    P2     P3  Peso MC
----------------------------------------------
D+1    2025-10-01      81.7  15.2   66.5    0.539
D+2    2025-10-02      79.8  14.8   64.9    0.441
D+3    2025-10-03      69.2  12.5   56.3    0.398
D+4    2025-10-04      52.1  10.6   40.7    0.506
D+5    2025-10-05      62.0  12.5   48.7    0.338
D+6    2025-10-06      85.4  15.8   69.4    0.535
D+7    2025-10-07      84.6  15.3   69.0    0.481

✅ previsoes_volume_mc.json exportado


## ✅ Notebook 03c — Concluído

### O que foi feito

**Split temporal (sem data leakage):**
- Treino: 2023-01-08 a 2025-09-30 (998 dias — Monte Carlo + real)
- Holdout: 2025-10-01 a 2025-12-31 (92 dias — 100% real, nunca visto)

**Modelos treinados:**
- v5 (lags: lag_1d, lag_7d, rolling_7d, rolling_30d) — Total, P2, P3
- v6 (v5 + is_dia_util) — Total, P2, P3

**Resultados — holdout out/2025:**

| Data | Real | v5 | v6 | Erro v5 | Erro v6 |
|---|---|---|---|---|---|
| 01/10 Qua | 81 | 81.3 | 82.1 | 0.3 ✅ | 1.1 |
| 02/10 Qui | 110 | 79.4 | 80.1 | 30.6 | 29.9 |
| 03/10 Sex | 98 | 69.2 | 69.2 | 28.8 | 28.8 |
| 04/10 Sáb | 41 | 52.7 | 51.5 | 11.7 | 10.5 |
| 05/10 Dom | 13 | 62.5 | 61.5 | 49.5 | 48.5 |
| 06/10 Seg | 77 | 85.1 | 85.7 | 8.1 | 8.7 |
| 07/10 Ter | 96 | 84.1 | 85.0 | 11.9 | 11.0 |
| **MAE** | | | | **20.1** | **19.8** |

**MAE cross-validation — Monte Carlo vs Original:**

| Horizonte | MC v6 | Original | Vencedor |
|---|---|---|---|
| D+1 | 14.62 | 17.06 | MC ✅ |
| D+2 | 12.80 | 10.08 | Original |
| D+3 | 12.48 | 8.25 | Original |
| D+4 | 12.01 | 12.31 | MC ✅ |
| D+5 | 18.88 | 9.66 | Original |
| D+6 | 18.08 | 20.84 | MC ✅ |
| D+7 | 15.27 | 14.14 | Original |

**Ensemble adaptativo por horizonte:**
- Pesos inversamente proporcionais ao MAE de cada modelo por horizonte
- MC favorito em D+1, D+4, D+6 | Original favorito em D+2, D+3, D+5, D+7
- Sem overfitting detectado (MAE holdout ≈ MAE cross-validation)

### Outputs gerados
- `outputs/previsoes_volume_mc.json` — previsão ensemble adaptativo D+1 a D+7 (total, P2, P3)

### Próximo passo
Notebook **04_xgboost_ola_risk.ipynb** — predição de violação de OLA por incidente

In [20]:
from sklearn.metrics import mean_absolute_error

# ── MAE Prophet nos 92 dias do holdout — rolling forecast ────────────────────
prophet_preds_92 = []

for i in range(len(serie_holdout_total)):
    futuro_1d = m_total_v6.make_future_dataframe(periods=i+1, freq='D').tail(1)
    lookup = serie_treino_total_lag.drop_duplicates('ds').set_index('ds')
    for r in REGS_V5:
        futuro_1d[r] = futuro_1d['ds'].map(lookup[r]).fillna(serie_treino_total_lag.tail(30)[r].mean())
    futuro_1d['is_dia_util'] = futuro_1d['ds'].dt.dayofweek.apply(lambda d: 0.0 if d >= 5 else 1.0)
    futuro_1d.loc[futuro_1d['ds'].dt.date.isin(feriados_set), 'is_dia_util'] = 0.0
    pred = m_total_v6.predict(futuro_1d)['yhat'].values[0]
    prophet_preds_92.append(max(0, pred))

prophet_preds_92 = np.array(prophet_preds_92)
reais_holdout_92 = serie_holdout_total['y'].values

mae_prophet_92 = mean_absolute_error(reais_holdout_92, prophet_preds_92)

print('=== MAE nos 92 dias completos do holdout ===')
print(f'Prophet v6:  {mae_prophet_92:.2f}')

=== MAE nos 92 dias completos do holdout ===
Prophet v6:  23.80
